In [ ]:
# import libraries and dependencies
# NOTE: run with the lidar_venv "Python 3 (ipykernel)" kernel (ROS-sourced) so rclpy + open3d both import.
import os
import sys
import time
import socket
import tempfile
import subprocess
from pathlib import Path

import numpy as np
import open3d as o3d
from IPython.display import Image, display

import rclpy
from sensor_msgs.msg import PointCloud2
from sensor_msgs_py import point_cloud2
from rclpy.qos import qos_profile_sensor_data

NB_DIR = Path.cwd()
PF_DIR = NB_DIR.parent
REPO_DIR = PF_DIR.parent
FUSION_DIR = PF_DIR / "fusion"
sys.path.insert(0, str(FUSION_DIR))
from classical import detect_people, remove_ground, DetectConfig  # noqa: E402

TOPIC = "/unilidar/cloud"  # PointCloud2 republished by the running unitree_lidar_ros2 node

# ---- viz mode ----------------------------------------------------------------
# USE_POPUP True  -> interactive, mouse-rotatable open3d windows on the VNC display
#                    (auto-starts headless Xorg + x11vnc via `make vnc-start` if needed).
# USE_POPUP False -> offscreen render shown inline as PNG (no display needed).
USE_POPUP = False
DISPLAY_ID = ":2"
VNC_PORT = 5901
# -----------------------------------------------------------------------------


def capture_frames(num_frames=1, timeout_s=60, topic=TOPIC):
    """Subscribe to the LiDAR topic and collect `num_frames` point clouds.

    Read-only: shares the stream the ROS2 node already publishes (no UDP port conflict).
    Returns a list of Nx5 float64 arrays: columns [x, y, z, intensity, ring].
    """
    frames = []
    if not rclpy.ok():
        rclpy.init()
    node = rclpy.create_node("nb_lidar_capture")

    def cb(msg):
        if len(frames) >= num_frames:
            return
        pc = point_cloud2.read_points(
            msg, field_names=("x", "y", "z", "intensity", "ring"), skip_nans=True
        )
        arr = np.column_stack(
            [pc["x"], pc["y"], pc["z"], pc["intensity"], pc["ring"]]
        ).astype(np.float64)
        frames.append(arr)

    node.create_subscription(PointCloud2, topic, cb, qos_profile_sensor_data)
    start = time.time()
    while len(frames) < num_frames and (time.time() - start) < timeout_s:
        rclpy.spin_once(node, timeout_sec=0.5)

    node.destroy_node()
    rclpy.shutdown()
    return frames


def _display_up(disp=DISPLAY_ID):
    return subprocess.run(
        ["xdpyinfo", "-display", disp], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    ).returncode == 0


def _vnc_up(port=VNC_PORT):
    s = socket.socket(); s.settimeout(0.3)
    try:
        s.connect(("127.0.0.1", port)); return True
    except OSError:
        return False
    finally:
        s.close()


def ensure_vnc():
    """Make sure headless Xorg :2 + x11vnc are up; start them via `make vnc-start` if not.

    Only restarts when actually down, so an active VNC session isn't disrupted. Points this
    process at DISPLAY=:2 so open3d windows land on the VNC desktop.
    """
    if _display_up() and _vnc_up():
        os.environ["DISPLAY"] = DISPLAY_ID
        os.environ["XAUTHORITY"] = ""
        print(f"[vnc] already up on {DISPLAY_ID} (port {VNC_PORT})")
        return
    print("[vnc] starting headless Xorg + x11vnc (make vnc-start)...")
    r = subprocess.run(["make", "vnc-start"], cwd=str(REPO_DIR), capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1000:]); print(r.stderr[-1000:])
        raise RuntimeError("make vnc-start failed")
    os.environ["DISPLAY"] = DISPLAY_ID
    os.environ["XAUTHORITY"] = ""
    print(f"[vnc] ready on {DISPLAY_ID} (view via VNC on port {VNC_PORT})")


def render_inline(geoms, width=960, height=680, title="", views=("iso", "top")):
    """Render open3d geometries offscreen (Vulkan/GPU, no display) and show inline as PNG."""
    r = o3d.visualization.rendering.OffscreenRenderer(width, height)
    r.scene.set_background([1.0, 1.0, 1.0, 1.0])

    cloud_pts = []
    for i, g in enumerate(geoms):
        mat = o3d.visualization.rendering.MaterialRecord()
        if isinstance(g, o3d.geometry.PointCloud):
            mat.shader = "defaultUnlit"; mat.point_size = 2.5
            pts = np.asarray(g.points)
            if len(pts):
                cloud_pts.append(pts)
        elif isinstance(g, o3d.geometry.AxisAlignedBoundingBox):
            ls = o3d.geometry.LineSet.create_from_axis_aligned_bounding_box(g)
            ls.paint_uniform_color(np.asarray(g.color) if np.any(g.color) else [1, 0, 0])
            g = ls; mat.shader = "unlitLine"; mat.line_width = 4.0
        elif isinstance(g, o3d.geometry.LineSet):
            mat.shader = "unlitLine"; mat.line_width = 2.0
        else:  # TriangleMesh (coordinate frame, etc.)
            mat.shader = "defaultUnlit"
        r.scene.add_geometry(f"g{i}", g, mat)

    if not cloud_pts:
        print("[render] nothing to show"); return
    allpts = np.vstack(cloud_pts)
    mn, mx = allpts.min(0), allpts.max(0)
    center = (mn + mx) / 2.0
    extent = float(np.linalg.norm(mx - mn)) or 1.0
    cam = {
        "iso": (center + np.array([extent * 0.7, -extent * 0.9, extent * 0.6]), [0, 0, 1]),
        "top": (center + np.array([1e-3, 1e-3, extent * 1.3]), [0, 1, 0]),
    }
    if title:
        print(title)
    for v in views:
        eye, up = cam[v]
        r.setup_camera(60.0, center.astype(np.float64), eye.astype(np.float64), np.asarray(up, np.float64))
        img = r.render_to_image()
        path = str(Path(tempfile.gettempdir()) / f"o3d_view_{v}.png")
        o3d.io.write_image(path, img)
        print(f"  [{v} view]")
        display(Image(filename=path))


def show_geoms(geoms, window_name="LiDAR", title="", views=("iso", "top")):
    """Unified display: interactive popup on VNC (USE_POPUP) or inline PNG otherwise."""
    if USE_POPUP:
        ensure_vnc()
        if title:
            print(title)
        print(f"[popup] opening interactive window on {DISPLAY_ID} - view in VNC; close window to continue")
        o3d.visualization.draw_geometries(geoms, window_name=window_name, width=1024, height=768)
    else:
        render_inline(geoms, title=title, views=views)


print("imports ok")
print("subscribing to:", TOPIC)
print("viz mode:", "interactive popup on " + DISPLAY_ID if USE_POPUP else "inline PNG")

# Shared state across cells
accumulated_points = None

In [ ]:
# take a lidar capture of a frame and display its data plus display it interactively
print("[lidar] waiting for one frame on", TOPIC, "...")
frames = capture_frames(num_frames=1, timeout_s=30)
if not frames:
    raise RuntimeError(f"no frame received on {TOPIC} within timeout (is the ROS2 node publishing?)")

pts = frames[0]
x, y, z, intensity, ring = pts[:, 0], pts[:, 1], pts[:, 2], pts[:, 3], pts[:, 4]

print(f"\n[points] shape={pts.shape} dtype={pts.dtype}")
print(f"[xyz] x=[{x.min():.2f}, {x.max():.2f}] y=[{y.min():.2f}, {y.max():.2f}] z=[{z.min():.2f}, {z.max():.2f}]")
print(f"[intensity] min={intensity.min():.1f} max={intensity.max():.1f} mean={intensity.mean():.1f}")
print(f"[rings] min={int(ring.min())} max={int(ring.max())}")

# Color by height (z): red = high, blue = low
pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts[:, :3]))
z_norm = (z - z.min()) / (z.max() - z.min() + 1e-6)
colors = np.zeros((pts.shape[0], 3))
colors[:, 0] = z_norm
colors[:, 2] = 1.0 - z_norm
pcd.colors = o3d.utility.Vector3dVector(colors)

show_geoms([pcd], window_name="Single LiDAR Frame", title="\n[single frame] height-colored")

In [ ]:
# take a set of lidar frames to display its data and the room interactively
NUM_FRAMES = 5
print(f"[lidar] capturing {NUM_FRAMES} frames from {TOPIC} ...")
frames = capture_frames(num_frames=NUM_FRAMES, timeout_s=60)
if not frames:
    raise RuntimeError("no frames captured")

for i, f in enumerate(frames):
    print(f"  frame {i+1}: {f.shape[0]:,} points")

# Merge frames (xyz only) -> denser room scan
accumulated_points = [f[:, :3] for f in frames]
cloud_xyz = np.vstack(accumulated_points)
print(f"\n[merged] {len(frames)} frames -> {cloud_xyz.shape[0]:,} total points")
print(f"[xyz bounds] x=[{cloud_xyz[:, 0].min():.2f}, {cloud_xyz[:, 0].max():.2f}] "
      f"y=[{cloud_xyz[:, 1].min():.2f}, {cloud_xyz[:, 1].max():.2f}] "
      f"z=[{cloud_xyz[:, 2].min():.2f}, {cloud_xyz[:, 2].max():.2f}]")

pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(cloud_xyz))
z = cloud_xyz[:, 2]
z_norm = (z - z.min()) / (z.max() - z.min() + 1e-6)
colors = np.zeros((cloud_xyz.shape[0], 3))
colors[:, 0] = z_norm
colors[:, 2] = 1.0 - z_norm
pcd.colors = o3d.utility.Vector3dVector(colors)

frame_axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
show_geoms([pcd, frame_axes], window_name="Multi-Frame Room View", title="\n[room scan] merged frames")

In [ ]:
# for the next few cells, run native functions and display til we do them all plus our custom functions for lidar etc
# Classical people detection: RANSAC ground removal -> HDBSCAN -> human-size filter (from fusion/classical.py)
cfg = DetectConfig(
    hdbscan_min_cluster_size=25,
    hdbscan_min_samples=12,
    min_cluster_points=25,
    height_min=0.8,
    height_max=2.2,
    footprint_max=0.9,
)

if not accumulated_points:
    print("[warning] no accumulated points; run the multi-frame cell above first")
else:
    cloud_xyz = np.vstack(accumulated_points)

    print("[classical] removing ground...")
    non_ground = remove_ground(cloud_xyz, cfg)
    print(f"  before: {cloud_xyz.shape[0]:,} points")
    print(f"  after:  {non_ground.shape[0]:,} points ({100*non_ground.shape[0]/cloud_xyz.shape[0]:.1f}%)")

    print("[classical] detecting people...")
    detections = detect_people(cloud_xyz, cfg=cfg, skip_ground=False)
    print(f"  {len(detections)} people detected")
    for i, det in enumerate(detections):
        print(f"    [{i+1}] centroid={np.round(det.centroid, 2)} "
              f"height={det.bbox_max[2]-det.bbox_min[2]:.2f}m "
              f"footprint={np.linalg.norm(det.bbox_max[:2]-det.bbox_min[:2]):.2f}m "
              f"points={det.n_points}")

    # non-ground points (green by height) + red detection boxes + axes
    geoms = []
    pcd_filtered = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(non_ground))
    zf = non_ground[:, 2]
    z_norm = (zf - zf.min()) / (zf.max() - zf.min() + 1e-6)
    colors_filtered = np.zeros((non_ground.shape[0], 3))
    colors_filtered[:, 1] = 0.3 + 0.7 * z_norm
    pcd_filtered.colors = o3d.utility.Vector3dVector(colors_filtered)
    geoms.append(pcd_filtered)

    for det in detections:
        bbox = o3d.geometry.AxisAlignedBoundingBox(det.bbox_min, det.bbox_max)
        bbox.color = [1, 0, 0]
        geoms.append(bbox)

    geoms.append(o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0]))
    show_geoms(geoms, window_name="Classical People Detection",
               title=f"\n[detections] {len(detections)} people (red boxes), green=non-ground")